**Data Processing **

Workflow

1.Itertrates theough all json files in the movement/folder


2.Extracts metadata like participant ID, session name and device location


3.Finds and loads .txt files from movement/timeseries/.


4.Merges all sensor data into single structured DataFrame for analysis


5.Saves the processed data as a CSV for machine Learning and Visualization.

Mount the google drive to Collab Notebook

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Load the original dataset from google drive and then load that dataset to the collab content

In [2]:
import zipfile
import os
import shutil

# 1. Define paths
# We get the zip from your Google Drive
drive_zip_path = '/content/drive/MyDrive/Parkinson_prediction/Parkinsons_zip.zip'
# We copy it to the fast, local Colab runtime
local_zip_path = '/content/Parkinsons_zip.zip'
# We extract it locally as well
local_extraction_destination = '/content/Parkinson_prediction_dataset)/'

# 2. Copy the zip file locally
print("Copying zip file from Google Drive to local environment...")
shutil.copy(drive_zip_path, local_zip_path)

# 3. Create local destination directory and extract
os.makedirs(local_extraction_destination, exist_ok=True)

print("Extracting files locally...")
with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
    zip_ref.extractall(local_extraction_destination)
print("Extraction complete! Your files are now ready at:", local_extraction_destination)

Copying zip file from Google Drive to local environment...
Extracting files locally...
Extraction complete! Your files are now ready at: /content/Parkinson_prediction_dataset)/


Import libraries

In [3]:
#import libraries

import os
import json
import pandas as pd
from glob import glob

Load and Parse json file. Then read time-series data from .txt file

In [4]:
#Define a function to load and parse json file for movement data and patient information data

def load_json(json_file):
  """ Load and parse a json file"""
  with open(json_file, 'r') as f:
    data = json.load(f)

  return data

In [5]:
#define a function to read the corresponding time-series data from the .txt file

def load_txt_data(txt_file):
  """ Load a .txt file and return a dataframe with sensor readings"""
  try:
    df = pd.read_csv(txt_file, header=None)
    df.columns = ["Time","Accel_X","Accel_Y","Accel_Z","Gyro_X","Gyro_Y","Gyro_Z"]
    return df

  except Exception as e:
    print(f"Error reading {txt_file}:{e}")
    return None

Optimization & Diagnostic Tools
Let's run a quick diagnostic check to see exactly how many files are found in your directories, verifying that the paths are correct and accessible.

In [6]:
import glob
import os

# Quick diagnostic to verify file count
obs_files = glob.glob(os.path.join("/content/Parkinson_prediction_dataset/parkinsons_zip/movement", "*.json"))
txt_files = glob.glob(os.path.join("/content/Parkinson_prediction_dataset/parkinsons_zip/timeseries", "*.txt"))
pat_files = glob.glob(os.path.join("/content/Parkinson_prediction_dataset/parkinsons_zip/patients", "*.json"))

print(f"Number of observation JSONs found: {len(obs_files)}")
print(f"Number of timeseries TXTs found: {len(txt_files)}")
print(f"Number of patient JSONs found: {len(pat_files)}")

Number of observation JSONs found: 0
Number of timeseries TXTs found: 0
Number of patient JSONs found: 0


Optimization & Diagnostic Tools
Let's run a quick diagnostic check to see exactly how many files are found in your directories, verifying that the paths are correct and accessible.

In [7]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import os

def fast_process_observations(observation_folder, txt_folder, patient_folder):
    # 1. Load patient metadata sequentially (usually few files)
    patient_metadata = {}
    patient_files = glob.glob(os.path.join(patient_folder, "*.json"))
    for file in patient_files:
        metadata = load_json(file)
        patient_metadata[metadata['id']] = {
            "Age": metadata["age"],
            "Height": metadata["height"],
            "Weight": metadata["weight"],
            "Gender": metadata["gender"],
            "Condition": metadata["condition"],
            "Age_At_Diagnosis": metadata.get("age_at_diagnosis", None),
            "Handedness": metadata["handedness"]
        }

    observation_files = glob.glob(os.path.join(observation_folder, "*.json"))
    combined_data = []

    # Helper function to process a single observation file in a thread
    def process_single_file(observation_file):
        try:
            metadata = load_json(observation_file)
            patient_id = metadata['subject_id']
            if patient_id not in patient_metadata:
                return []

            patient_info = patient_metadata[patient_id]
            local_dfs = []

            for session in metadata.get("session", []):
                task = session["record_name"]
                for record in session["records"]:
                    device_location = record["device_location"]
                    file_name = record["file_name"].split("/")[-1]
                    txt_file_path = os.path.join(txt_folder, file_name)

                    if os.path.exists(txt_file_path):
                        df = load_txt_data(txt_file_path)
                        if df is not None:
                            df["Patient_id"] = patient_id
                            df["Task"] = task
                            df["Device location"] = device_location
                            for key, value in patient_info.items():
                                df[key] = value
                            local_dfs.append(df)
            return local_dfs
        except Exception as e:
            print(f"Error processing {observation_file}: {e}")
            return []

    # Use ThreadPoolExecutor to load files in parallel (highly speeds up drive latency)
    print("Starting parallel load...")
    with ThreadPoolExecutor(max_workers=8) as executor:
        results = executor.map(process_single_file, observation_files)

    for result_list in results:
        combined_data.extend(result_list)

    if combined_data:
        return pd.concat(combined_data, ignore_index=True)
    else:
        print("No valid data files found")
        return None

# Folder paths
observation_folder = "/content/drive/MyDrive/Parkinson_prediction/movement"
txt_folder = "/content/drive/MyDrive/Parkinson_prediction/timeseries"
patient_folder = "/content/drive/MyDrive/Parkinson_prediction/patients"


# Run the optimized function
pd_df = fast_process_observations(observation_folder, txt_folder, patient_folder)

Starting parallel load...


In [8]:
#save the merged dataset

if pd_df is not None:
  #pd_df.to_csv("/content/drive/MyDrive/Parkinson_prediction/parkinsons_data1.csv", index=False)
  print("Data processing complete. Saved as parkinsons-data.csv")

else:
  print("Could not merge dataset due to missig files")

Data processing complete. Saved as parkinsons-data.csv


In [9]:
#Display top 10 processed data

pd_df.head(10)

,Time,Accel_X,Accel_Y,Accel_Z,Gyro_X,Gyro_Y,Gyro_Z,Patient_id,Task,Device location,Age,Height,Weight,Gender,Condition,Age_At_Diagnosis,Handedness
0,0.000000,-0.003958,0.002360,1.746120e-03,-0.015492,0.005616,0.001034,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
1,0.009903,-0.004005,0.002259,8.355394e-04,-0.012276,0.005605,-0.003218,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
2,0.019901,0.000833,0.002225,9.437218e-04,-0.009051,0.002432,-0.002137,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
3,0.029907,-0.000191,0.004109,4.783000e-07,-0.008911,0.002154,-0.000931,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
4,0.039984,0.001769,0.004064,-1.902400e-03,-0.007830,0.000034,0.000145,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
5,0.050019,0.003715,0.005005,7.265950e-05,-0.004608,-0.003129,0.002288,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
6,0.059944,0.004753,0.005929,-9.233341e-04,-0.005622,-0.010597,0.004460,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
7,0.072727,0.001953,0.002909,-9.748325e-04,-0.005624,-0.011639,0.007654,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
8,0.080140,0.001115,0.000862,-4.249670e-05,-0.008827,-0.012707,0.010851,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right
9,0.090037,-0.001712,0.002680,8.749380e-04,-0.007776,-0.009520,0.007642,001,Relaxed,LeftWrist,56,173,78,male,Healthy,56,right


In [12]:
df['Condition'].value_counts()

,count
Condition,
Parkinson's,121440
Healthy,34760
Other Movement Disorders,26400
Essential Tremor,12320
Atypical Parkinsonism,6600
Multiple Sclerosis,4840


Extract Sample Data

In [10]:
#Extract sample data
#sort by time (if time column exists)

df = pd_df.sort_values(by=["Patient_id","Task","Device location","Time"])

#'Extract the first 20 rows per group

df = df.groupby(["Patient_id","Task","Device location","Gender"],group_keys=False).head(20)

#save the extracted data
#df.to_csv("/content/drive/MyDrive/Parkinson_prediction/parkinsons_reduced_data.csv")

